# Smart Classroom - Step 6: train PhysNet (3D CNN) on UBFC-rPPG and compare with POS

1. **Runtime -> Change runtime type -> T4 GPU**
2. UBFC-rPPG **DATASET_2** must be in your Google Drive (folder with `subject1/vid.avi`, `subject1/ground_truth.txt`, ...).
3. Your latest code must be pushed to GitHub (`git push`): cell 2 clones it.
4. Run the cells in order. Preprocessing is saved to Drive, so it runs only once.
5. Download `physnet_results.zip` and unzip it **into** your `SmartClassroom` folder.

In [ ]:
# 1. Google Drive + paths  (EDIT UBFC if your folder has another name)
from google.colab import drive
drive.mount('/content/drive')
UBFC = '/content/drive/MyDrive/DATASET_2'           # shortcut to DATASET_2 (contains subject1, subject3, ...)
CACHE = '/content/drive/MyDrive/ubfc_cache'         # preprocessed subjects (~30 MB each), kept on Drive
import os
print(sorted(os.listdir(UBFC))[:5], '...', len(os.listdir(UBFC)), 'entries')

In [ ]:
# 2. Code from GitHub + MediaPipe (face landmarks for the crops and the POS regions)
!rm -rf smartClassroom && git clone -q https://github.com/sofiensgh/smartClassroom.git
%cd smartClassroom
!pip install -q mediapipe
import torch; print(torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: change the runtime type!')

In [ ]:
# 3. Preprocess: face crops 72x72 + POS skin colour + ground-truth PPG, one .npz per subject
#    (about 1 min per subject; already-processed subjects are skipped, so you can re-run after a disconnect)
!python -u -m smartclass.rppg.preprocess --ubfc "{UBFC}" --out "{CACHE}"

In [ ]:
# 4. Train PhysNet (split by subject) and evaluate PhysNet vs POS on the test subjects
!python -u -m smartclass.training.train_physnet --cache "{CACHE}" --epochs 30

In [ ]:
# 5. Results
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('output/heart/results.csv'))
print(open('output/heart/split.txt').read())
for f in ['physnet_curves', 'scatter', 'bland_altman', 'example_wave']:
    display(Image(f'output/heart/{f}.png'))

In [ ]:
# 6. Download: unzip INTO your SmartClassroom folder (it contains models/physnet.pt and output/heart/)
!zip -qr ../physnet_results.zip models/physnet.pt output/heart
from google.colab import files
files.download('../physnet_results.zip')